# Alouette-2 Image Inventory

#### Updated: July 2026

## Imports

In [ ]:
from pathlib import Path
import os
import pandas as pd

## Helper Functions

In [ ]:
def count_num_subdirectories(directory_path : str):
    """
    Count the number of immediate subdirectories within a specified directory.

    Parameters
    ----------
    directory_path : str
        Path to the directory whose direct subdirectories will be counted.

    Returns
    -------
    num_subdirectories : int
        Number of immediate subdirectories contained in the specified
        directory.

    Notes
    -----
    Only direct child directories are counted. Nested subdirectories within
    those directories are not included in the count.
    """
    directories = os.listdir(directory_path)
    subdirectories = [subdirectory for subdirectory in directories if os.path.isdir(os.path.join(directory_path, subdirectory))]

    num_subdirectories = len(subdirectories)
    return num_subdirectories

In [4]:
def list_subdirectories(directory_path: str):
    """
    Retrieve the names of all immediate subdirectories within a specified
    directory.

    Parameters
    ----------
    directory_path : str
        Path to the directory whose direct subdirectories will be listed.

    Returns
    -------
    subdirectories : list[str]
        List containing the names of all immediate subdirectories found in
        the specified directory.

    Notes
    -----
    Only direct child directories are included. Nested subdirectories within
    those directories are not recursively searched or returned.
    """
    directories = os.listdir(directory_path)
    subdirectories = [subdirectory for subdirectory in directories if os.path.isdir(os.path.join(directory_path, subdirectory))]
    return subdirectories

In [5]:
def count_png_files(directory_path :str):
    """
    Count the total number of PNG files within a directory and all of its
    subdirectories.

    Parameters
    ----------
    directory_path : str
        Path to the root directory that will be searched for PNG files.

    Returns
    -------
    png_count : int
        Total number of files with a '.png' extension found in the directory
        tree. File extension matching is case-insensitive.

    Notes
    -----
    The search includes all nested subdirectories beneath the specified
    directory. Files with extensions such as '.PNG' or '.Png' are counted
    because filenames are converted to lowercase before comparison.
    """
    png_count = 0
    for root, dirs, files in os.walk(directory_path):
        for file in files: 
            if file.lower().endswith(".png"):
                png_count += 1
    return png_count


In [6]:
def get_subdirectory_with_max_pngs(pngs_per_subdirectory: list):
    """
    Identify the subdirectory or subdirectories containing the maximum number
    of PNG images.

    Parameters
    ----------
    pngs_per_subdirectory : list[dict]
        List of dictionaries containing PNG inventory information:
        [
            {
                'Subdirectory': relative path (str),
                'PNG Image Count': integer (int)
            },
            ...
        ]

    Returns
    -------
    max_png_subdirectory : list[str]
        List of subdirectories that share the highest PNG count.
    max_val : int
        Maximum number of PNG files found in a subdirectory.

    Implementation Details
    ----------------------
    Iterates through the inventory records, tracking the largest PNG count
    encountered. If multiple subdirectories have the same maximum count,
    all are returned.
    """
    
    max_png_subdirectory = []
    max_val = -1

    for dir in pngs_per_subdirectory:
        val = dir['PNG Image Count']
        subdir =  dir['Subdirectory']

        if val == max_val:
            max_png_subdirectory.append(subdir)
        
        elif val > max_val:
            max_val = val
            max_png_subdirectory = [subdir]

    return max_png_subdirectory, max_val

def get_subdirectory_with_min_pngs(pngs_per_subdirectory: list):
    """
    Identify the subdirectory or subdirectories containing the minimum number
    of PNG images.

    pngs_per_subdirectory : list[dict]
        List of dictionaries containing PNG inventory information:

        [
            {
                'Subdirectory': relative path (str),
                'PNG Image Count': integer
            },
            ...
        ]

    Returns
    -------
    min_png_subdirectory : list[str]
        List of subdirectories that share the lowest PNG count.
    min_val : int
        Minimum number of PNG files found in a subdirectory.

    Implementation Details
    ----------------------
    Iterates through the inventory records, tracking the smallest PNG count.
    If multiple subdirectories have the same minimum count,
    all are returned.
    """
    min_png_subdirectory = []
    min_val = float('inf')

    for dir in pngs_per_subdirectory:
        val = dir['PNG Image Count']
        subdir =  dir['Subdirectory']

        if val == min_val:
            min_png_subdirectory.append(subdir)
        
        elif val < min_val:
            min_val = val
            min_png_subdirectory = [subdir]

    return min_png_subdirectory, min_val


In [7]:
def print_summary(title, base_dir, num_subdirectories, total_num_pngs, max_png_val, min_png_val, max_png_subdirectory, min_png_subdirectory, inventory_df):
    """
    Print a formatted summary of PNG image inventory statistics for a base
    directory and its subdirectories.

    Parameters
    ----------
    title : str
        Title displayed at the top of the summary report.
    base_dir : str
        Path to the base directory being inventoried.
    num_subdirectories : int
        Total number of immediate subdirectories within the base directory.
    total_num_pngs : int
        Total number of PNG files across all inventoried directories (includes PNGs in nested subdirectories).
    max_png_val : int
        Highest PNG count found in any subdirectory.
    min_png_val : int
        Lowest PNG count found in any subdirectory.
    max_png_subdirectory : list[str]
        Subdirectories containing the maximum PNG count.
    min_png_subdirectory : list[str]
        Subdirectories containing the minimum PNG count.
    inventory_df : pandas.DataFrame
        DataFrame containing PNG inventory details for each subdirectory.
        Each row represents a subdirectory and includes:
            - 'Subdirectory': Relative path of the subdirectory.
            - 'PNG Image Count': Number of PNG files found in that subdirectory (including PNGs in nested subdirectories).

    Returns
    -------
    None
        The function prints the summary directly to the console.
    """
    # Bold formatter
    BOLD = "\033[1m"
    END = "\033[0m"
    UNDERLINE = "\033[4m"

    print(f"{BOLD}{UNDERLINE}\n========================= {title} ========================={END}\n")

    print(f"{BOLD}Base Directory:{END} {base_dir}")
    print(f"{BOLD}Total Number of Subdirectories in Base Directory:{END} {num_subdirectories}")
    print(f"{BOLD}Total PNG Files in Base Directory and All Subdirectories:{END} {total_num_pngs}\n")

    print(f"{BOLD}Maximum PNGs in a Subdirectory:{END} {max_png_val}", "  |  ", f"{BOLD}Subdirectories with Maximum PNGs:{END} {max_png_subdirectory}\n")

    print(f"{BOLD}Minimum PNGs in a Subdirectory:{END} {min_png_val}", "  |  ", f"{BOLD}Subdirectories with Minimum PNGs:{END} {min_png_subdirectory}\n")

    print(f"{BOLD}PNG Files by Subdirectory:{END}")
    print("----------------------------------------")

    print(inventory_df.to_string(index=False))  

## Imageupload_20260428 - Alouette II Inventory

In [8]:

def get_AlouetteII_Inventory(base_dir):
    num_subdirectories = count_num_subdirectories(base_dir)
    subdirectories = list_subdirectories(base_dir)
    total_pngs = count_png_files(base_dir)

    pngs_per_subdirectory = [
                                {
                                    'Subdirectory': f"{subdirectory}",
                                    'PNG Image Count': count_png_files(base_dir+'/'+subdirectory)
                                }

                                for subdirectory in subdirectories
                            ]
    
    inventory_df = pd.DataFrame(pngs_per_subdirectory).reset_index(drop=True)

    max_png_subdirectory, max_val = get_subdirectory_with_max_pngs(pngs_per_subdirectory)
    min_png_subdirectory, min_val = get_subdirectory_with_min_pngs(pngs_per_subdirectory)

    # Printing the Inventory
    print_summary(
        title='Alouette II Inventory',
        base_dir=base_dir,
        num_subdirectories=num_subdirectories,
        total_num_pngs=total_pngs,
        min_png_val=min_val,
        max_png_val=max_val,
        min_png_subdirectory=min_png_subdirectory,
        max_png_subdirectory=max_png_subdirectory,
        inventory_df=inventory_df
    )
    


In [ ]:
get_AlouetteII_Inventory(base_dir="L:/DATA/ISIS/Imageupload_20260428/CSA_2026-04-07/2/SSA-23")


========================= Alouette II Inventory =========================

Base Directory: L:/DATA/ISIS/Imageupload_20260428/CSA_2026-04-07/2/SSA-23
Total Number of Subdirectories in Base Directory: 72
Total PNG Files in Base Directory and All Subdirectories: 19009

Maximum PNGs in a Subdirectory: 340   |   Subdirectories with Maximum PNGs: ['23-017', '23-063']

Minimum PNGs in a Subdirectory: 36   |   Subdirectories with Minimum PNGs: ['23-016']

PNG Files by Subdirectory:
----------------------------------------
Subdirectory  PNG Image Count
      23-001              311
      23-002              269
      23-003              326
      23-004               47
      23-005              318
      23-006              295
      23-007              329
      23-008              287
      23-009              311
      23-010              318
      23-011              296
      23-012              323
      23-013              317
      23-014              300
      23-015              328